# Data Loading — FastF1

**Goal:** pull historical F1 race results (2018–2026) into one tidy DataFrame and cache it to `data/raw/` for the features/model notebooks.

**Approach:** learn the FastF1 API on a single race first, then generalize to a season/round loop.

In [14]:
import fastf1
import pandas as pd
import numpy as np

from f1pred import paths

# FastF1 caches raw API responses to disk -> huge speedup on re-runs.
CACHE_DIR = paths.DATA / "cache" / "fastf1"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
fastf1.Cache.enable_cache(str(CACHE_DIR))

paths.ensure_dirs()
print("cache:", CACHE_DIR)

cache: C:\Users\tusha\Documents\Project\f1-pred-modal\data\cache\fastf1


## 1. Load one session (learn the API)

Start with a single known race and inspect what FastF1 returns before scaling up.

In [15]:
# One race: 2024 season, round 1 (Bahrain), Race session = "R".
session = fastf1.get_session(2024, 1, "R")
session.load()   # first run downloads (slow); afterwards it's cached
# Things to inspect:
#   session.results  -> DataFrame, one row per driver
#                       (DriverId, TeamId, GridPosition, Position, Points, Status...)
#   session.event    -> event metadata (EventName, RoundNumber, EventDate...)
results = session.results  # DataFrame, one row per driver
results.head()



core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '16', '63', '4', '44', '81', '14', '18', '24', '20', '3', '22', '23', '27', '31', '10', '77', '2']


,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,...,Position,ClassifiedPosition,GridPosition,Q1,Q2,Q3,Time,Status,Points,Laps
1,1,M VERSTAPPEN,VER,max_verstappen,Red Bull Racing,3671c6,red_bull,Max,Verstappen,Max Verstappen,...,1.0,1,1.0,NaT,NaT,NaT,0 days 01:31:44.742000,Finished,26.0,57.0
11,11,S PEREZ,PER,perez,Red Bull Racing,3671c6,red_bull,Sergio,Perez,Sergio Perez,...,2.0,2,5.0,NaT,NaT,NaT,0 days 00:00:22.457000,Finished,18.0,57.0
55,55,C SAINZ,SAI,sainz,Ferrari,e8002d,ferrari,Carlos,Sainz,Carlos Sainz,...,3.0,3,4.0,NaT,NaT,NaT,0 days 00:00:25.110000,Finished,15.0,57.0
16,16,C LECLERC,LEC,leclerc,Ferrari,e8002d,ferrari,Charles,Leclerc,Charles Leclerc,...,4.0,4,2.0,NaT,NaT,NaT,0 days 00:00:39.669000,Finished,12.0,57.0
63,63,G RUSSELL,RUS,russell,Mercedes,27f4d2,mercedes,George,Russell,George Russell,...,5.0,5,3.0,NaT,NaT,NaT,0 days 00:00:46.788000,Finished,10.0,57.0


In [16]:
# One race: 2024 season, round 1 (Bahrain), Race session = "R".
session = fastf1.get_session(2024, 1, "R")
session.load()   # first run downloads (slow); afterwards it's cached
# Things to inspect:
#   session.results  -> DataFrame, one row per driver
#                       (DriverId, TeamId, GridPosition, Position, Points, Status...)
#   session.event    -> event metadata (EventName, RoundNumber, EventDate...)
results = session.results  # DataFrame, one row per driver
results.head()



core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '16', '63', '4', '44', '81', '14', '18', '24', '20', '3', '22', '23', '27', '31', '10', '77', '2']


,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,...,Position,ClassifiedPosition,GridPosition,Q1,Q2,Q3,Time,Status,Points,Laps
1,1,M VERSTAPPEN,VER,max_verstappen,Red Bull Racing,3671c6,red_bull,Max,Verstappen,Max Verstappen,...,1.0,1,1.0,NaT,NaT,NaT,0 days 01:31:44.742000,Finished,26.0,57.0
11,11,S PEREZ,PER,perez,Red Bull Racing,3671c6,red_bull,Sergio,Perez,Sergio Perez,...,2.0,2,5.0,NaT,NaT,NaT,0 days 00:00:22.457000,Finished,18.0,57.0
55,55,C SAINZ,SAI,sainz,Ferrari,e8002d,ferrari,Carlos,Sainz,Carlos Sainz,...,3.0,3,4.0,NaT,NaT,NaT,0 days 00:00:25.110000,Finished,15.0,57.0
16,16,C LECLERC,LEC,leclerc,Ferrari,e8002d,ferrari,Charles,Leclerc,Charles Leclerc,...,4.0,4,2.0,NaT,NaT,NaT,0 days 00:00:39.669000,Finished,12.0,57.0
63,63,G RUSSELL,RUS,russell,Mercedes,27f4d2,mercedes,George,Russell,George Russell,...,5.0,5,3.0,NaT,NaT,NaT,0 days 00:00:46.788000,Finished,10.0,57.0


## 2. Extract the columns you care about

Build one tidy *row-per-driver-per-race* frame by **merging the Race and Qualifying sessions** on `DriverId`:

- **from Race:** `DriverId`, `TeamId`, finishing `Position` / `ClassifiedPosition`, `GridPosition`
- **from Qualifying:** quali `Position` and `Q3` time
- **plus** `Points`, `Status` (DNF reason), and `Abbreviation` (readable driver code)

Gotchas to keep in mind: `GridPosition == 0.0` means a pit-lane start (not P0); `Q3` is `NaT` for anyone knocked out in Q1/Q2 (only the top 10 run Q3); `ClassifiedPosition` is a string (`"1"`…`"20"`, `"R"` retired, `"D"` disqualified).

In [17]:
def tidy_results(race, quali) -> pd.DataFrame:
    """One tidy row per driver for a race weekend: race result + qualifying, merged on driver."""
    ev = race.event

    # --- race side ---
    r = race.results[
        [
            "DriverId",
            "Abbreviation",        # readable code (VER, HAM) — handy for eyeballing
            "TeamId",
            "GridPosition",        # 0.0 == pit-lane start (clean later; don't treat as P0)
            "Position",            # final finishing place (float; NaN if not classified)
            "ClassifiedPosition",  # str: "1".."20", "R" retired, "D" disqualified
            "Points",
            "Status",              # "Finished" / "+1 Lap" / "Accident" / "Engine" ... -> DNF reason
        ]
    ].copy()
    
    r = r.rename(
        columns={
            "GridPosition": "grid_position",
            "Position": "race_position",
            "Points": "points",
            "Status": "status",
        }
    )

    # --- qualifying side ---
    q = quali.results[["DriverId", "Position" ,"Q1", "Q2", "Q3"]].copy()
    q["best_quali"] = q[["Q1", "Q2", "Q3"]].min(axis=1)
    q = q.rename(columns={"Position": "quali_position", "Q3": "q3_time", "Q2": "q2_time", "Q1": "q1_time"})


    # --- merge (race is the anchor: every driver who started the GP) ---
    df = r.merge(q, on="DriverId", how="left")

    # --- race identifiers, so rows stay self-describing after concat ---
    df["season"] = int(ev["EventDate"].year)
    df["round"] = int(ev["RoundNumber"])
    df["event"] = ev["EventName"]

    return df


quali = fastf1.get_session(2024, 1, "Q"); quali.load()
tidy_results(session, quali).head()

core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '63', '55', '11', '14', '4', '81', '44', '27', '22', '18', '23', '3', '20', '77', '24', '2', '31', '10']


,DriverId,Abbreviation,TeamId,grid_position,race_position,ClassifiedPosition,points,status,quali_position,q1_time,q2_time,q3_time,best_quali,season,round,event
0,max_verstappen,VER,red_bull,1.0,1.0,1,26.0,Finished,1.0,0 days 00:01:30.031000,0 days 00:01:29.374000,0 days 00:01:29.179000,0 days 00:01:29.179000,2024,1,Bahrain Grand Prix
1,perez,PER,red_bull,5.0,2.0,2,18.0,Finished,5.0,0 days 00:01:30.221000,0 days 00:01:29.932000,0 days 00:01:29.537000,0 days 00:01:29.537000,2024,1,Bahrain Grand Prix
2,sainz,SAI,ferrari,4.0,3.0,3,15.0,Finished,4.0,0 days 00:01:29.909000,0 days 00:01:29.573000,0 days 00:01:29.507000,0 days 00:01:29.507000,2024,1,Bahrain Grand Prix
3,leclerc,LEC,ferrari,2.0,4.0,4,12.0,Finished,2.0,0 days 00:01:30.243000,0 days 00:01:29.165000,0 days 00:01:29.407000,0 days 00:01:29.165000,2024,1,Bahrain Grand Prix
4,russell,RUS,mercedes,3.0,5.0,5,10.0,Finished,3.0,0 days 00:01:30.350000,0 days 00:01:29.922000,0 days 00:01:29.485000,0 days 00:01:29.485000,2024,1,Bahrain Grand Prix


## 3. Loop over seasons and rounds

Filter each season's schedule to races that have **already happened**, so nothing fails — no `try/except` needed.

FastF1's API allows 500 requests/hour. If a run stops on a rate-limit error, wait ~30–60 min and re-run: already-fetched races come from the local cache for free, so it finishes over a couple of passes.

In [19]:
SEASONS = range(2022, 2027)
LOAD_KW = dict(laps=False, telemetry=False, weather=False, messages=False)  # results only -> fewer API calls

frames = []
for season in SEASONS:
    schedule = fastf1.get_event_schedule(season, include_testing=False)
    past = schedule[schedule["EventDate"] < pd.Timestamp.now()]   # only races that have run
    for rnd in past["RoundNumber"]:
        race = fastf1.get_session(season, int(rnd), "R")
        race.load(**LOAD_KW)
        quali = fastf1.get_session(season, int(rnd), "Q")
        quali.load(**LOAD_KW)
        frames.append(tidy_results(race, quali))
        print(f"ok  {season} R{rnd}")

races = pd.concat(frames, ignore_index=True)
print(races.shape)

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['16', '55', '44', '63', '20', '77', '31', '22', '14', '24', '47', '18', '23', '3', '4', '6', '27', '11', '1', '10']
core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['16', '1', '55', '11', '44', '77', '20', '14', '63', '10', '31', '47', '4', '23', '24', '22', '27', '3', '18', '6']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


ok  2022 R1


core           INFO 	Finished loading data for 20 drivers: ['1', '16', '55', '11', '63', '31', '4', '10', '20', '44', '24', '27', '18', '23', '77', '14', '3', '6', '22', '47']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Qualifying [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['11', '16', '55', '1', '31', '63', '14', '77', '10', '20', '4', '3', '24', '47', '18', '44', '23', '27', '6', '22']
core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


ok  2022 R2


core           INFO 	Finished loading data for 20 drivers: ['16', '11', '63', '44', '4', '3', '31', '77', '10', '23', '24', '18', '47', '20', '22', '6', '14', '1', '5', '55']
core           INFO 	Loading data for Australian Grand Prix - Qualifying [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['16', '1', '11', '4', '44', '63', '3', '31', '55', '14', '10', '77', '22', '24', '47', '23', '20', '5', '6', '18']
core           INFO 	Loading data for Emilia Romagna Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


...

ok  2026 R7


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 22 drivers: ['63', '3', '12', '81', '44', '6', '1', '16', '30', '41', '5', '27', '10', '87', '43', '31', '23', '14', '18', '55', '11', '77']
core           INFO 	Loading data for Austrian Grand Prix - Qualifying [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
logger      WARNING 	Failed to load result data from Ergast!
core        WARNING 	No result data for this session available on Ergast! (This is expected 

ok  2026 R8


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
logger      WARNING 	Failed to load result data from Ergast!
core        WARNING 	No result data for this session available on Ergast! (This is expected for recent sessions)
core           INFO 	Finished loading data for 22 drivers: ['12', '16', '44', '63', '6', '1', '3', '81', '41', '30', '5', '27', '87', '55', '10', '23', '31', '77', '43', '11', '14', '18']
core           INFO 	Loading data for British Grand Prix - Qualifying [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has bee

ok  2026 R9
(3638, 16)


## 4. Save to `data/raw/`

Persist as parquet so the features/model notebooks just read it back.

In [ ]:
out = paths.RAW / "races.parquet"
races.to_parquet(out, index=False)
print("wrote", out, races.shape)